In [0]:
%sql
CREATE TABLE IF NOT EXISTS identifier(:catalog || '.silver.suppliers_scd2') (
    supplier_id INT,
    supplier_name STRING,
    contact_email STRING,
    country STRING,
    effective_date DATE,
    end_date DATE,
    is_current BOOLEAN,
    version INT
)

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW suppliers_raw_view AS
SELECT supplier_id, supplier_name, coalesce(contact_email, 'not provided') as contact_email, country,
       ingestion_date
FROM (
    SELECT *,
           row_number() OVER (PARTITION BY supplier_id ORDER BY ingestion_date DESC) AS rn
    FROM identifier(:catalog || '.bronze.suppliers_raw')
    WHERE supplier_id IS NOT NULL
)
WHERE rn = 1;

In [0]:
%sql
MERGE INTO identifier(:catalog || '.silver.suppliers_scd2') t
USING suppliers_raw_view s
ON t.supplier_id = s.supplier_id
   AND t.is_current = true
   AND (
        t.supplier_name <> s.supplier_name
        OR t.contact_email <> s.contact_email
        OR t.country <> s.country
   )
WHEN MATCHED THEN UPDATE SET
    t.end_date = s.ingestion_date,
    t.is_current = false;

In [0]:
CREATE OR REPLACE TEMPORARY VIEW suppliers_with_version AS
SELECT
    s.supplier_id,
    s.supplier_name,
    s.contact_email,
    s.country,
    s.ingestion_date,
    COALESCE(MAX(t.version), 0) + 1 AS next_version
FROM suppliers_raw_view s
LEFT JOIN identifier(:catalog || '.silver.suppliers_scd2') t
    ON s.supplier_id = t.supplier_id
GROUP BY s.supplier_id, s.supplier_name, s.contact_email, s.country, s.ingestion_date;

In [0]:
%sql
MERGE INTO identifier(:catalog || '.silver.suppliers_scd2') t
USING suppliers_with_version s
ON t.supplier_id = s.supplier_id AND t.is_current = true
WHEN NOT MATCHED THEN INSERT (
    supplier_id, supplier_name, contact_email, country,
    effective_date, end_date, is_current, version
) VALUES (
    s.supplier_id, s.supplier_name, s.contact_email, s.country,
    s.ingestion_date, NULL, true, s.next_version
);